# Linear Algebra

### NEUBEH/PBIO 545 — Quantitative Methods in Neuroscience

*Adapted from* [`matlab/LinearAlgebra.m`](../matlab/LinearAlgebra.m) by Fred Rieke.

This tutorial introduces the basic concepts of linear algebra that the rest of the course
leans on. The aims are:

1. learn to **visualize** operations in linear algebra — what different matrix
   transformations actually do to a vector;
2. introduce the key **types** of linear transformation and their properties;
3. identify **"natural" coordinate systems** (eigenvectors) for describing a system;
4. illustrate these ideas with concrete examples, ending with a three-state ion channel.

| Part | Topic |
|---|---|
| I | Representing data as vectors |
| II | $n$ equations in $n$ unknowns, and the matrix inverse |
| III | Invertible transformations: rotation, reflection, scaling |
| IV | Projection matrices, rank, column space and null space |
| V | The eigensystem and "natural" coordinates |
| VI | *(optional)* Markov chains and state-transition matrices |

Run it cell by cell (**Shift+Enter**). The text is part of the tutorial, not decoration —
the homework questions in particular are where the ideas get consolidated.

If you need a review of the basic concepts (what a vector is, how matrix multiplication
works) read the Linear Algebra Primer from Eero Simoncelli on the course website, or
Strang, *Linear Algebra and its Applications*.

---

### A note on reading this alongside the MATLAB original

If you have the `.m` file open next to this notebook, a handful of translation traps are
worth knowing about up front. Each one is flagged again at the point where it first bites.

| MATLAB | Python / NumPy |
|---|---|
| `A * B` is **matrix** multiply | `A * B` is **elementwise**; use `A @ B` |
| `v(1)` is the first element | `v[0]` is the first element (0-based) |
| `M \ v` solves $Mu=v$ | `np.linalg.solve(M, v)` |
| `inv(M) * v` | works, but `solve` is more accurate and faster — prefer it |
| `[V, D] = eig(M)` → eigenvalues in a **diagonal matrix** `D` | `w, V = np.linalg.eig(M)` → eigenvalues in a 1-D **array** `w` |
| `[U,S,V] = svd(A)` returns $V$ | `U, s, Vt = np.linalg.svd(A)` returns $V^{\mathsf{T}}$ |
| `orth(M)`, `null(M)`, `rank(M)` built in | `scipy.linalg.orth`, `scipy.linalg.null_space`, `np.linalg.matrix_rank` |
| `M^N` is the **matrix** power | `M ** N` is elementwise; use `np.linalg.matrix_power(M, N)` |

And one that is not a language difference at all but a mathematical fact people forget:
**eigenvector sign and scale are arbitrary.** If $Me = \lambda e$ then $M(ce) = \lambda(ce)$
for any $c \neq 0$. MATLAB and NumPy both return unit-length eigenvectors, but the sign
they pick is whatever the underlying LAPACK routine happened to produce. Never write code
that depends on it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as sla
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the '3d' projection)

rng = np.random.default_rng(545)   # fixed seed, so every run reproduces the figures

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "font.size": 10,
})
np.set_printoptions(precision=4, suppress=True)

# ---------------------------------------------------------------------------
# Ports of the MATLAB helper files the original tutorial depends on.
#   PlotVector.m   -> plot_vector
#   Plot3DVector.m -> plot_3d_vector
# The MATLAB originals just draw a line from the origin to the tip of the
# vector; here we add an arrowhead and an optional label, because a plot of
# five overlapping line segments is hard to read.
# ---------------------------------------------------------------------------

def plot_vector(v, color="b", ax=None, label=None, lw=2, ls="-"):
    '''Draw the 2-D vector v as an arrow from the origin. (PlotVector.m)'''
    ax = ax or plt.gca()
    v = np.asarray(v, float).ravel()
    ax.annotate("", xy=(v[0], v[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                linestyle=ls, shrinkA=0, shrinkB=0))
    # A zero-length line carries the legend entry: annotations cannot.
    ax.plot([], [], color=color, lw=lw, ls=ls, label=label)
    return ax


def plot_3d_vector(v, color="b", ax=None, label=None, lw=2, ls="-"):
    '''Draw the 3-D vector v as a line from the origin, tipped with a dot.

    (Plot3DVector.m.) Matplotlib's 3-D axes have no reliable arrowhead, so we
    mark the tip with a filled circle instead -- it survives rotation, which a
    hand-drawn cone does not.
    '''
    ax = ax or plt.gca()
    v = np.asarray(v, float).ravel()
    ax.plot([0, v[0]], [0, v[1]], [0, v[2]], color=color, lw=lw, ls=ls,
            label=label)
    ax.scatter([v[0]], [v[1]], [v[2]], color=color, s=30, depthshade=False)
    return ax


def make_3d_axes(fig, pos=111, lim=(-1, 1), elev=12, azim=42, labels="xyz"):
    '''A 3-D axis with equal limits on all three axes and the MATLAB view angle.

    Plot3DVector.m calls view(42, 12). MATLAB's view(az, el) and matplotlib's
    view_init(elev, azim) take their arguments in the opposite order and use a
    different azimuth zero, so the pictures will not be pixel-identical -- the
    geometry is what matters.
    '''
    ax = fig.add_subplot(pos, projection="3d")
    if lim is not None:
        lo, hi = lim
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_zlim(lo, hi)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel(labels[0], labelpad=1); ax.set_ylabel(labels[1], labelpad=1)
    ax.set_zlabel(labels[2], labelpad=1)
    ax.set_box_aspect((1, 1, 1))
    # Matplotlib's default 3-D tick density overlaps badly at these figure
    # sizes; four ticks per axis keeps everything readable.
    for a in (ax.xaxis, ax.yaxis, ax.zaxis):
        a.set_major_locator(plt.MaxNLocator(4))
    ax.tick_params(labelsize=7, pad=0)
    return ax


def edge_on_view(basis, tilt=9.0):
    '''(elev, azim) that looks nearly along a plane, so it appears as a sliver.

    A plane appears edge-on when the camera's viewing direction lies IN the
    plane. Take any in-plane vector d and convert it to matplotlib's spherical
    (elev, azim) convention. Guessing angles by hand -- which is what the
    MATLAB original does with its hard-coded view(12,56) -- almost never lands
    exactly edge-on.
    '''
    d = basis[:, 0] / np.linalg.norm(basis[:, 0])
    elev = np.degrees(np.arcsin(np.clip(d[2], -1, 1)))
    azim = np.degrees(np.arctan2(d[1], d[0]))
    # A perfectly edge-on camera also looks straight down one of matplotlib's
    # axis panes and renders an unreadable box, so tilt a few degrees off. The
    # MATLAB original does the same thing by hand ("this is not quite in the
    # right plane, so vectors do not obscure each other").
    return elev + tilt, azim


def draw_plane(ax, basis, half=1.6, color="0.6", alpha=0.18):
    '''Shade the plane spanned by the two columns of `basis` (a 3x2 array).'''
    s = np.linspace(-half, half, 2)
    S, T = np.meshgrid(s, s)
    P = S[..., None] * basis[:, 0] + T[..., None] * basis[:, 1]
    ax.plot_surface(P[..., 0], P[..., 1], P[..., 2],
                    color=color, alpha=alpha, shade=False)

---
## Part I. Representing data as vectors

Linear algebra is built out of vectors and matrices. A **vector** is simply an ordered
collection of numbers. The ordering matters because it lets us associate each number with a
particular direction or axis. You already do this whenever you plot a point in Cartesian
coordinates as $(x, y, z)$.

The idea is completely general, and the axes need not be spatial. Starbucks could represent
espresso drinks as pairs of numbers — one for the amount of milk, one for the amount of
espresso:

$$\mathrm{cap} = \begin{pmatrix} 0.5 \\ 0.5\end{pmatrix},\qquad
  \mathrm{espresso} = \begin{pmatrix} 0 \\ 1\end{pmatrix},\qquad
  \mathrm{latte} = \begin{pmatrix} 0.7 \\ 0.3\end{pmatrix}$$

(This is at least how it *should* be — before you could choose different types of milk,
different syrups, and so on. Those would add more axes.)

These are **column** vectors. We could equally use row vectors; the distinction matters
mainly once matrices enter the picture. We will mostly use columns.

The advantage of representing data as vectors is that it lets us *visualize* it (as long as
the dimension is low enough), and — as we will see — lets us operate on it with matrices,
which is often both efficient and intuitive.

In [ ]:
cap      = np.array([0.5, 0.5])
espresso = np.array([0.0, 1.0])
latte    = np.array([0.7, 0.3])

fig, ax = plt.subplots(figsize=(5, 5))
plot_vector(cap,      "tab:blue",   ax, label="cap")
plot_vector(espresso, "tab:red",    ax, label="espresso")
plot_vector(latte,    "k",          ax, label="latte")
ax.set(xlim=(-0.05, 1.0), ylim=(-0.05, 1.1), aspect="equal",
       xlabel="milk fraction", ylabel="espresso fraction",
       title="Espresso drinks as 2-D vectors")
ax.legend(loc="upper right")
fig.tight_layout()

One thing this example points out is that the units on the two axes do not have to be the
same. It is also good for small talk next time you are at Starbucks.

**Matrices are operators that transform one vector into another.** They can scale the data
along one or more axes, rotate it to a different set of axes, project it onto a subset of
the axes, or map from one space (milk–espresso) into a different one (sleep–wake). The rest
of the tutorial is devoted to these transformations.

---
## Part II. $n$ equations in $n$ unknowns, and the matrix inverse

One use of linear algebra — usually the first you meet in a real linear algebra course — is
solving systems of linear equations. For example, what $(x, y, z)$ satisfies

$$\begin{aligned}
2x + 3y - 3z &= 1\\
-x + y \phantom{{}- 3z} &= 2\\
3x - y + z &= -1
\end{aligned}$$

These are *linear* because $x$, $y$ and $z$ appear only to the first power (no $x^2$, no
$\sqrt{x}$) and are never multiplied by each other (no $xy$, no $yz$).

Coupled linear equations have a solution as long as there are as many equations as unknowns
(three of each here) **and** the equations are independent — i.e. no equation is a weighted
sum of the others. That is the case above. When either assumption fails there is either no
exact solution or infinitely many. All is not necessarily lost: we can often find a useful
approximate solution using the singular value decomposition (see the SVD tutorial).

One way to solve this is by substitution: solve one equation for one variable, substitute
into the other two, repeat. A second, far more systematic way is to write the whole system
as a single **matrix equation**

$$M\,u = v$$

where $M$ is $3\times 3$ and $u$, $v$ are 3-element column vectors.

In [ ]:
M = np.array([[ 2,  3, -3],
              [-1,  1,  0],
              [ 3, -1,  1]], dtype=float)

v = np.array([1.0, 2.0, -1.0])

print("M =\n", M)
print("v =", v)
print("\nRow 1 of M times a trial u gives 2x + 3y - 3z -- check this against the "
      "first equation above.")

$u$ is the unknown vector containing $x$, $y$ and $z$. Make sure you understand how this
notation is equivalent to the three equations above. If it is obscure, consider the simpler
case: if $M = \begin{pmatrix}2 & 0\\ 0 & 3\end{pmatrix}$ and
$v = \begin{pmatrix}4\\6\end{pmatrix}$, what is $u$?

What we want is the vector $u$ that $M$ transforms into $v$. It helps to think about this
visually. $v$ is a vector in a 3-D space, and $M$ is an operator on that space; our problem
is to find the vector that $M$ carries onto $v$.

In [ ]:
fig = plt.figure(figsize=(5.5, 5))
ax = make_3d_axes(fig, lim=(-1.5, 2.5))
plot_3d_vector(v, "tab:blue", ax, label="v")
ax.set_title("The target vector $v$")
ax.legend()
fig.tight_layout()

### Guessing does not work

Let's try to find $u$ by trial and error. Start with $u = (1, 0, 0)$.

In [ ]:
# --- guess 1 -----------------------------------------------------------
# NOTE: in MATLAB `M * u` is matrix multiplication. In NumPy `*` is
# ELEMENTWISE. The matrix product is `M @ u`. This is the single most common
# porting bug, and for square M it fails silently -- you get a wrong answer,
# not an error.
u = np.array([1.0, 0.0, 0.0])

fig = plt.figure(figsize=(5.5, 5))
ax = make_3d_axes(fig, lim=(-1.5, 3.2))
plot_3d_vector(v,     "tab:blue",  ax, label="v (target)")
plot_3d_vector(u,     "tab:green", ax, label="u (guess)")
plot_3d_vector(M @ u, "tab:red",   ax, label="M u")
ax.set_title("Guess 1:  u = (1, 0, 0)")
ax.legend(loc="upper left")
fig.tight_layout()

print("M @ u =", M @ u, "   but we want v =", v)

Our guess $u$ is in green, $Mu$ in red, and the target $v$ in blue. We are a long way off.
Here is another attempt.

In [ ]:
# --- guess 2 -----------------------------------------------------------
u = np.array([0.0, 2.0, 1.0])

fig = plt.figure(figsize=(5.5, 5))
ax = make_3d_axes(fig, lim=(-1.5, 3.2))
plot_3d_vector(v,     "tab:blue",  ax, label="v (target)")
plot_3d_vector(u,     "tab:green", ax, label="u (guess)")
plot_3d_vector(M @ u, "tab:red",   ax, label="M u")
ax.set_title("Guess 2:  u = (0, 2, 1)")
ax.legend(loc="upper left")
fig.tight_layout()

print("M @ u =", M @ u, "   want v =", v)
print("residual ||M u - v|| = %.4f" % np.linalg.norm(M @ u - v))

That is closer — but I cheated, and picked a form for $u$ after looking at the answer. At
this point you are probably ready to abandon the whole linear-algebra approach. If not, try
a few more guesses to convince yourself this is not a fruitful strategy.

So we want something systematic. That is what the **matrix inverse** gives us. If $M$
transforms $u$ into $v$, then $M^{-1}$ undoes the transformation: it takes $v$ back to $u$.

### Checking that the inverse works as advertised

Start with a vector $u$, apply $M$ to make $v = Mu$, then apply $M^{-1}$ and confirm we get
$u$ back. All three panels below use **identical axis limits** so the lengths are directly
comparable.

In [ ]:
u = np.array([1.0, 1.0, 1.0])
Minv = np.linalg.inv(M)

fig = plt.figure(figsize=(12, 4))
lim = (-0.5, 3.2)
for k, (vec, col, ttl) in enumerate([
        (u,            "tab:green", "$u$"),
        (M @ u,        "tab:red",   "$Mu$"),
        (Minv @ M @ u, "k",         "$M^{-1}Mu$")], start=1):
    ax = make_3d_axes(fig, 130 + k, lim=lim)
    plot_3d_vector(vec, col, ax)
    ax.set_title(ttl + "  = (%.2f, %.2f, %.2f)" % tuple(vec))
fig.tight_layout()

print("inv(M) @ M =\n", Minv @ M)
print("\nsame thing, without the print-precision rounding:")
print(np.array2string(Minv @ M, formatter={"float": lambda x: "%+.3e" % x}))
print("\nis it EXACTLY the identity?      ", bool(np.all(Minv @ M == np.eye(3))))
print("is it the identity to tolerance? ", np.allclose(Minv @ M, np.eye(3)))
print("largest deviation: %.2e" % np.max(np.abs(Minv @ M - np.eye(3))))

On the left is the original $u$, in the middle $Mu$, and on the right $M^{-1}Mu$ — back
where we started. You can see why directly from the product $M^{-1}M$ printed above: it is
the **identity matrix**, ones on the diagonal and zeros everywhere else. The identity simply
transforms a vector back into itself.

Look at the second printout, though. The off-diagonal entries are not exactly zero — they
are of order $10^{-16}$, and the `==` test fails. That is floating-point rounding, not
mathematics. `np.allclose` is the right way to test such a thing; `==` is not. This will
recur throughout the course.

Everything here assumes the inverse *exists*, which we return to in Part IV.

### Back to the three equations

Multiply both sides of $Mu = v$ by $M^{-1}$:

$$M^{-1}M\,u = M^{-1}v \quad\Longrightarrow\quad u = M^{-1}v$$

since $M^{-1}Mu = u$. This helps enormously: the problem becomes "apply $M^{-1}$ to $v$",
and the result is exactly the $x, y, z$ you would have got by brute-force substitution.

⚠️ **Do not actually form the inverse.** MATLAB's own documentation says the same thing:
`M \ v` is preferred over `inv(M) * v`. The NumPy equivalent of `\` is
`np.linalg.solve(M, v)`. It factorizes $M$ and back-substitutes, which is both faster and
numerically better behaved than computing $M^{-1}$ and multiplying. We show both below only
to demonstrate they agree.

In [ ]:
u_inv   = Minv @ v                    # MATLAB: inv(M) * v
u_solve = np.linalg.solve(M, v)       # MATLAB: M \ v      <-- prefer this

print("u via inv(M) @ v :", u_inv)
print("u via solve      :", u_solve)
print("agree            :", np.allclose(u_inv, u_solve))

print("\ncheck -- M @ u should equal v")
print("M @ u =", M @ u_solve, "    v =", v)
print("exact solution is (-2/11, 20/11, 15/11) =",
      np.array([-2/11, 20/11, 15/11]))

### Doing it from scratch: Gaussian elimination

`solve` is a black box. The mechanism inside it *is* the lesson here — it is exactly the
substitute-and-eliminate procedure you would do by hand, organized so a computer can do it.
Let's write it out, with partial pivoting (always eliminate using the largest available
pivot, which is what keeps the arithmetic stable), and check it against NumPy.

In [ ]:
def gauss_solve(A, b):
    '''Solve A u = b by Gaussian elimination with partial pivoting.

    This is the algorithm behind MATLAB's `\` and np.linalg.solve. Written out
    so you can see there is no magic: forward-eliminate to upper triangular,
    then back-substitute.
    '''
    A = np.array(A, dtype=float)          # copy: we modify in place
    b = np.array(b, dtype=float)
    n = len(b)

    for k in range(n):                    # 0-BASED: MATLAB would write 1:n
        # --- pivot: swap in the row with the largest entry in column k
        p = k + np.argmax(np.abs(A[k:, k]))
        if np.abs(A[p, k]) < 1e-12:
            raise np.linalg.LinAlgError("matrix is singular -- no unique solution")
        if p != k:
            A[[k, p]] = A[[p, k]]
            b[[k, p]] = b[[p, k]]
        # --- eliminate column k from every row below (vectorized)
        f = A[k+1:, k] / A[k, k]
        A[k+1:, k:] -= np.outer(f, A[k, k:])
        b[k+1:]     -= f * b[k]

    # --- back-substitution
    u = np.zeros(n)
    for k in range(n - 1, -1, -1):
        u[k] = (b[k] - A[k, k+1:] @ u[k+1:]) / A[k, k]
    return u


u_scratch = gauss_solve(M, v)
print("from scratch :", u_scratch)
print("np.linalg.solve:", np.linalg.solve(M, v))
print("max abs difference: %.2e" % np.max(np.abs(u_scratch - np.linalg.solve(M, v))))
print("\ndet(M) = %.4f  (nonzero, so M is invertible and the solution is unique)"
      % np.linalg.det(M))

> ### Homework question 1
> **(a)** Why does the identity transform a vector back into itself? Pick some examples and
> convince yourself this is true.
>
> **(b)** Confirm that the $u$ we obtained above is right by the brute-force substitution
> approach applied to the original coupled linear equations.
>
> **(c)** Why does checking that $Mu = v$ tell us we found the right solution $u$?
>
> **(d)** Modify `gauss_solve` to skip the pivoting step (always use `A[k, k]` as the
> pivot). Find a $3\times 3$ system on which the unpivoted version gives a visibly worse
> answer than the pivoted one. *Hint: make one diagonal entry very small but not zero.*

---
## Part III. Invertible transformations: rotation, reflection, scaling

So far we have used matrices to solve equations. Now we look at what matrices *do*
geometrically. This part covers three kinds of **invertible** transformation: rotation,
reflection and scaling.

All three are **linear operators**, which means they satisfy two conditions. First,
additivity:

$$L(u + v) = L(u) + L(v)$$

— applying the operator to a sum equals the sum of applying it to each piece. Second,
homogeneity:

$$L(cu) = c\,L(u)$$

for any scalar $c$. Every matrix multiplication satisfies both; every operation satisfying
both can be written as a matrix multiplication.

**Invertible** transformations are, as the name suggests, ones that can be undone: the
matrix generating the transformation has an inverse. Part IV looks at non-invertible ones.

### Rotation matrices

Rotations preserve the *length* of a vector but change its *direction*. A given rotation
matrix rotates every vector through the same angle about the same axis. In two dimensions,

$$R(\theta) = \begin{pmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{pmatrix}$$

Start with a vector in a 2-D space.

In [ ]:
v = np.array([0.5, 0.8])

# The MATLAB original writes `Angle = 2 * 3.14159 * 40/360` and calls the result
# "steradians" -- it means RADIANS (a steradian is a unit of solid angle).
# In Python just use np.deg2rad, and np.pi rather than a truncated 3.14159.
angle = np.deg2rad(40)
R = np.array([[np.cos(angle), -np.sin(angle)],
              [np.sin(angle),  np.cos(angle)]])

Rv = R @ v

fig, ax = plt.subplots(figsize=(5, 5))
th = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(th)*np.linalg.norm(v), np.sin(th)*np.linalg.norm(v),
        ":", color="0.6", lw=1, label="circle of constant length")
plot_vector(v,  "tab:blue", ax, label="original $v$")
plot_vector(Rv, "tab:red",  ax, label="$Rv$ (rotated 40$\\degree$)")
ax.set(xlim=(-1, 1), ylim=(-1, 1), aspect="equal",
       xlabel="x component", ylabel="y component",
       title="A 40$\\degree$ rotation in the plane")
ax.legend(loc="lower left", fontsize=8)
fig.tight_layout()

print("R =\n", R)
print("|v|  = %.4f" % np.linalg.norm(v))
print("|Rv| = %.4f   (rotation preserves length)" % np.linalg.norm(Rv))
print("angle between v and Rv = %.2f degrees"
      % np.degrees(np.arccos(v @ Rv / (np.linalg.norm(v)*np.linalg.norm(Rv)))))

Both vectors sit on the dotted circle: the rotation moved the tip around the circle without
changing its distance from the origin.

This is not restricted to 2-D. Here is a rotation in 3-D about the $z$ axis:

$$R_z(\theta) = \begin{pmatrix}\cos\theta & -\sin\theta & 0\\
\sin\theta & \cos\theta & 0\\ 0 & 0 & 1\end{pmatrix}$$

Look at the structure. The upper-left $2\times 2$ block is exactly the 2-D rotation matrix,
so the action on the $x$ and $y$ coordinates is the same as before. The zeros in the last
*column* mean the $x$ and $y$ components of the output pick up no part of the input's $z$
component. The zeros in the last *row* mean the $z$ output picks up no part of $x$ or $y$.
The 1 in the corner preserves the $z$ component. Hence: rotation about $z$.

In [ ]:
v = np.array([1.0, -0.5, 1.0])

angle = np.deg2rad(40)
Rz = np.array([[np.cos(angle), -np.sin(angle), 0],
               [np.sin(angle),  np.cos(angle), 0],
               [0,              0,             1]])
Rv = Rz @ v

# Left: an oblique 3-D view. Right: looking straight down the z axis, which is
# MATLAB's `view(0, 90)`. Matplotlib's 3-D axes render that top-down view with
# a mangled z-axis, so we draw it honestly as a plain 2-D plot of the x and y
# components -- which is exactly what "looking down z" means.
fig = plt.figure(figsize=(11, 5))
ax = make_3d_axes(fig, 121, lim=(-1.5, 1.5))
plot_3d_vector(v,  "tab:blue", ax, label="original $v$")
plot_3d_vector(Rv, "tab:red",  ax, label="$R_z v$")
ax.set_title("oblique 3-D view")
ax.legend(loc="upper left", fontsize=8)

ax2 = fig.add_subplot(122)
plot_vector(v[:2],  "tab:blue", ax2, label="original $v$")
plot_vector(Rv[:2], "tab:red",  ax2, label="$R_z v$")
th = np.linspace(0, 2*np.pi, 200)
r_xy = np.linalg.norm(v[:2])
ax2.plot(r_xy*np.cos(th), r_xy*np.sin(th), ":", color="0.6", lw=1)
ax2.set(xlim=(-1.5, 1.5), ylim=(-1.5, 1.5), aspect="equal", xlabel="x", ylabel="y",
        title="looking down the $z$ axis (the $x$-$y$ components)")
ax2.legend(loc="upper left", fontsize=8)
fig.tight_layout()

print("v    =", v, "  |v|    = %.4f" % np.linalg.norm(v))
print("Rz v =", Rv, "  |Rz v| = %.4f" % np.linalg.norm(Rv))
print("z component unchanged:", np.isclose(v[2], Rv[2]))

From the $z$-axis view (right panel) you see exactly what happened in the $x$–$y$ plane; the
$z$ component, unchanged, points straight at you.

### Rotations are invertible

If we rotate by $+40\degree$ and then by $-40\degree$ we get back where we started. Here we
know how to construct the inverse explicitly — just flip the sign of the angle. Usually we
would have to compute it numerically.

In [ ]:
v = np.array([1.0, -0.5, 1.0])

angle = np.deg2rad(-40)
Rz_inv = np.array([[np.cos(angle), -np.sin(angle), 0],
                   [np.sin(angle),  np.cos(angle), 0],
                   [0,              0,             1]])

Rv     = Rz @ v
RinvRv = Rz_inv @ Rv

fig = plt.figure(figsize=(12, 4))
lim = (-1.5, 1.5)
for k, (vec, col, ttl) in enumerate([
        (v,      "tab:blue",  "original $v$"),
        (Rv,     "tab:red",   "$R_z v$  (+40$\\degree$)"),
        (RinvRv, "tab:green", "$R_z^{-1}R_z v$  (back)")], start=1):
    ax = make_3d_axes(fig, 130 + k, lim=lim)
    plot_3d_vector(vec, col, ax)
    ax.set_title(ttl, fontsize=10)
fig.tight_layout()

print("hand-built -40 deg matrix:\n", Rz_inv)
print("\nnumerical inv(Rz):\n", np.linalg.inv(Rz))
print("\nthe two agree:", np.allclose(Rz_inv, np.linalg.inv(Rz)))
print("Rz @ inv(Rz) is the identity:", np.allclose(Rz @ np.linalg.inv(Rz), np.eye(3)))
print("\nFor a rotation the inverse is just the transpose:",
      np.allclose(np.linalg.inv(Rz), Rz.T))

Notice the last line: for a rotation matrix the inverse *is* the transpose. That is the
defining property of an **orthogonal** matrix, and it makes rotations exceptionally cheap to
undo. It will come back in Part V and in the SVD tutorial.

### Reflection matrices

Reflection also preserves length. The simplest example is reflection about the $x$ axis —
flip the sign of the $y$ coordinate:

$$F = \begin{pmatrix}1 & 0\\ 0 & -1\end{pmatrix}$$

In [ ]:
F = np.array([[1.0, 0.0],
              [0.0, -1.0]])

pairs = [np.array([0.3, 0.9]), np.array([-0.3, 0.9])]

fig, ax = plt.subplots(figsize=(5, 5))
for i, a in enumerate(pairs):
    plot_vector(a,     "tab:blue", ax, label="original" if i == 0 else None)
    plot_vector(F @ a, "tab:red",  ax, label="reflected" if i == 0 else None)
ax.axhline(0, color="0.7", lw=1)
ax.set(xlim=(-1, 1), ylim=(-1, 1), aspect="equal",
       xlabel="x component", ylabel="y component",
       title="Reflection about the $x$ axis")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()

print("det(F) = %.1f   -- a reflection has determinant -1" % np.linalg.det(F))
print("det(R) = %.1f   -- a rotation has determinant +1" % np.linalg.det(R))

In each case the red vector is identical to the blue except its $y$ coordinate has been
multiplied by $-1$. Convince yourself that **no single rotation** can account for the change
in *both* vectors: the right-hand pair would need a clockwise rotation and the left-hand
pair a counter-clockwise one. (Make sure you consider the sign of your rotation angle.)

The determinants make this precise. A rotation has determinant $+1$ — it preserves
orientation (handedness). A reflection has determinant $-1$ — it reverses it. No product of
rotations can ever produce a determinant of $-1$.

### Scaling matrices

Scaling matrices stretch or contract one or more axes but do not *mix* them the way a
rotation does. The identity matrix is our first (and particularly boring) example — it
applies a scale of 1 to every axis.

Here is a better one. Take a two-dimensional cloud of data points with mean zero and
standard deviations 0.5 along $x$ and 1.5 along $y$.

In [ ]:
# 2 x 5000: each COLUMN is one (x, y) data point, matching the MATLAB layout.
# (Note this is the transpose of the row=observation convention np.cov expects
#  by default -- see the PCA tutorial for that trap.)
Dist = np.vstack([rng.normal(0, 0.5, 5000),
                  rng.normal(0, 1.5, 5000)])

# We want to measure distance from the mean in units of STANDARD DEVIATIONS.
# Build a matrix with the SDs along the diagonal, then divide by it -- i.e.
# apply its inverse. For a diagonal matrix the inverse is just the reciprocals
# on the diagonal, which is exactly "divide each axis by its own sd".
Msd = np.array([[0.5, 0.0],
                [0.0, 1.5]])

NewDist = np.linalg.inv(Msd) @ Dist

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.8))
for ax, D, xl, yl, ttl in [
        (axes[0], Dist,    "amp 1", "amp 2",             "raw data"),
        (axes[1], NewDist, "amp 1 / sd 1", "amp 2 / sd 2", "rescaled by the sds")]:
    ax.plot(D[0], D[1], ".", ms=1.5, alpha=0.4, color="tab:blue")
    ax.set(xlim=(-5, 5), ylim=(-5, 5), aspect="equal",
           xlabel=xl, ylabel=yl, title=ttl)     # identical limits: comparable
fig.tight_layout()

print("nominal sds: [0.5   1.5  ]")
print("sds before :", Dist.std(axis=1).round(3))
print("sds after  :", NewDist.std(axis=1).round(3))

Now the standard deviation along each axis is 1 — or very nearly so: the printout shows
about 0.97 and 0.99 rather than exactly 1, because the *sample* standard deviations of 5000
random draws are not exactly the nominal 0.5 and 1.5 we divided by. (Divide by the measured
sds instead of the nominal ones and you get exactly 1. Which you should do depends on
whether you know the true scale or have to estimate it.)

Either way the cloud is now symmetric, and moving a distance of 1 along either axis moves
you by one standard deviation.

Such normalization is useful in many settings. A neuroscience example is **adaptation**.
Suppose you are interested in how a cell responds to three different temporal frequencies,
and you represent the response as a point in a 3-D space with one axis per frequency. You
then deliver an adapting stimulus at one frequency. If adaptation changes the cell's gain at
*that* frequency only, you can account for its whole effect by scaling that one axis. If
adaptation crosses over to affect gain at the other frequencies too, you will need to scale
those axes as well.

> ### Homework question 2
> **(a)** Why do rotation matrices take the form above (start with the 2-D case)? Draw a
> picture and use the definitions of sine and cosine from trigonometry.
>
> **(b)** What would the matrix look like for a rotation about the $x$ axis of $30\degree$
> followed by a rotation about the $z$ axis of $45\degree$?
>
> **(c)** When a rotation and a scaling are both applied to a vector, does the order matter?
> Can you show this? *(You now have the tools to just try it — build $RS$ and $SR$ and
> compare.)*
>
> **(d)** Why do we use the *inverse* of `Msd` above as the scaling matrix? Why does that
> inverse take the form it does?

---
## Part IV. Projection matrices, rank, column space and null space

Above we considered only invertible matrices. The columns of an invertible matrix are
**independent**. Note that independent does not mean *orthogonal* — it simply means that no
column can be written as a linear combination of the others.

In the $n$-equations-in-$n$-unknowns problem we required the equations to be independent to
get a unique solution; if they are not, the problem is underdetermined. The same thing shows
up as a matrix property: if a matrix has dependent columns, its inverse does not exist. The
reason is that such a matrix does not map each vector to a *unique* other vector — it
**projects many vectors onto one**. Let's look at that directly.

### An example of a non-invertible matrix

The simplest example is a matrix that projects every vector onto the $x$–$y$ plane, i.e.
one that annihilates the $z$ component:

$$N = \begin{pmatrix}1&0&0\\0&1&0\\0&0&0\end{pmatrix}$$

In [ ]:
N = np.array([[1., 0, 0],
              [0, 1., 0],
              [0, 0, 0]])

v1 = np.array([1.0, 0.5, 1.0])
v2 = np.array([1.0, 0.5, 0.2])

u1, u2 = N @ v1, N @ v2

fig = plt.figure(figsize=(11, 5))
lim = (-0.2, 1.2)
ax = make_3d_axes(fig, 121, lim=lim)
plot_3d_vector(v1, "tab:blue", ax, label="$v_1$")
plot_3d_vector(v2, "tab:red",  ax, label="$v_2$")
ax.set_title("two distinct vectors"); ax.legend(fontsize=9)

ax = make_3d_axes(fig, 122, lim=lim)
plot_3d_vector(u1, "tab:blue", ax, lw=5, label="$Nv_1$")
plot_3d_vector(u2, "tab:red",  ax, lw=2, ls="--", label="$Nv_2$")
ax.set_title("after projection: identical"); ax.legend(fontsize=9)
fig.tight_layout()

print("N @ v1 =", u1)
print("N @ v2 =", u2)
print("identical:", np.allclose(u1, u2))
print("det(N) = %.1f -> not invertible" % np.linalg.det(N))

They land exactly on top of each other — the right panel draws $Nv_2$ as a dashed line on
top of the thicker solid $Nv_1$ so you can see both. $N$ has collapsed two different vectors
onto the same point. There is therefore no way to invert the transformation: given the
output, we cannot recover what the $z$ component *was*.

This property is not restricted to matrices that delete a coordinate axis. A matrix can
project every vector onto a plane sitting at an angle in the space:

$$M = \begin{pmatrix}0.5 & 1 & 0\\ -0.5 & 0.5 & -0.75\\ 1 & 0 & 1\end{pmatrix}$$

⚠️ The MATLAB original plots the projected vectors inside `axis([0 1 0 1 0 1])`, but for
these inputs the outputs are $Nv_1 = (1, -1, 2)$ and $Nv_2 = (1, -0.4, 1.2)$ — both run well
outside that box, so the figure gets clipped and does not show what its comment claims.
We use limits that actually contain the data.

In [ ]:
Mproj = np.array([[ 0.5, 1.0,  0.0],
                  [-0.5, 0.5, -0.75],
                  [ 1.0, 0.0,  1.0]])

u1, u2 = Mproj @ v1, Mproj @ v2

# The column space of Mproj is a plane; shade it so the claim is visible rather
# than something you have to take on faith. (orth() is built up from scratch in
# the next cell; here we just use the SciPy version to draw the picture.)
basis = sla.orth(Mproj)

fig = plt.figure(figsize=(11, 5))
lim = (-2.2, 2.2)
views = [(12, 42, "oblique view"),
         edge_on_view(basis) + ("nearly edge-on: the plane is a thin sliver",)]
for k, (elev, azim, ttl) in enumerate(views, start=1):
    ax = make_3d_axes(fig, 120 + k, lim=lim, elev=elev, azim=azim)
    draw_plane(ax, basis, half=2.2, color="0.35", alpha=0.30)
    plot_3d_vector(u1, "tab:blue", ax, label="$Mv_1$")
    plot_3d_vector(u2, "tab:red",  ax, label="$Mv_2$")
    ax.set_title(ttl, fontsize=10)
    ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()

print("M @ v1 =", u1)
print("M @ v2 =", u2)
print("det(M) = %.2e  (zero to rounding -> not invertible)" % np.linalg.det(Mproj))

The two projected vectors are different — this matrix is not simply deleting an axis — but
they both lie in the **same plane**. Tilt the view until you are looking nearly along that
plane (right panel) and the whole shaded surface collapses to a thin sliver, with both
vectors inside it. That plane is the column space of $M$, which we now define properly.

### Column space and null space

A *space* is described by a set of axes. The $x$–$y$ plane is spanned by the $x$ and $y$
axes — or equivalently by the axes $x+y$ and $x-y$, or by many other pairs. Two spaces
attached to a matrix are especially useful:

- the **column space**: the set of all vectors reachable as $Mu$ for some $u$ — equivalently,
  all linear combinations of the columns of $M$;
- the **null space**: the set of vectors $u$ with $Mu = 0$ — the directions the matrix
  destroys.

Each has a dimension: the number of independent axes needed to describe it. And they are
tied together by the **rank–nullity theorem**:

$$\dim(\text{column space}) + \dim(\text{null space}) = \text{number of columns}$$

Matrices also have a **row space** — exactly what you would guess, the space spanned by the
rows. The dimension of the row space equals the dimension of the column space, for *every*
matrix including non-square ones. That common number is the **rank**.

In [ ]:
def my_orth_null(A, tol=None):
    '''Orthonormal bases for the column space and the null space of A, from the SVD.

    Ports MATLAB's orth() and null(). This IS how MATLAB and SciPy implement
    them -- the SVD hands you both spaces at once.

    ** MATLAB's [U, S, V] = svd(A) returns V.
       NumPy's  U, s, Vt = np.linalg.svd(A) returns V TRANSPOSED. **
    So the null-space vectors are ROWS of Vt, not columns.
    '''
    U, s, Vt = np.linalg.svd(A)
    if tol is None:
        tol = max(A.shape) * np.finfo(float).eps * (s[0] if s.size else 0.0)
    r = int((s > tol).sum())                 # the rank
    col_space  = U[:, :r]                    # first r left singular vectors
    null_space = Vt[r:, :].T                 # remaining RIGHT singular vectors
    return col_space, null_space, r, s


col_M, null_M, rank_M, sv_M = my_orth_null(Mproj)
print("singular values of M      :", sv_M)
print("rank(M) from scratch      :", rank_M)
print("np.linalg.matrix_rank(M)  :", np.linalg.matrix_rank(Mproj))
print("\ncolumn space basis (from scratch):\n", col_M)
print("scipy.linalg.orth agrees (same plane):",
      np.allclose(col_M @ col_M.T, sla.orth(Mproj) @ sla.orth(Mproj).T))
print("\nnull space basis (from scratch):\n", null_M)
print("scipy.linalg.null_space:\n", sla.null_space(Mproj))
print("\nM @ (null vector) =", (Mproj @ null_M).ravel(), " -- annihilated, as promised")
print("rank + nullity = %d + %d = %d = number of columns"
      % (rank_M, null_M.shape[1], rank_M + null_M.shape[1]))

⚠️ **Sign again.** `sla.null_space` and our `my_orth_null` may return null vectors pointing
in opposite directions, and the two column-space bases may be rotated within the plane. Both
answers are equally correct: a basis for a space is not unique. That is why the check above
compares the **projectors** $BB^{\mathsf{T}}$ — which depend only on the space — rather than
the basis vectors themselves.

So $M$ has rank 2: its column space is a plane, which is exactly the plane we shaded in the
figure. If $M$ were an invertible $3\times3$ matrix it would have rank 3; if it had only one
independent column, rank 1.

The point is clearer still for the simple "project onto the $x$–$y$ plane" matrix $N$.

In [ ]:
col_N, null_N, rank_N, sv_N = my_orth_null(N)
print("rank(N) =", rank_N)
print("column space of N:\n", col_N)
print("-> two unit vectors along x and y (up to sign / rotation within the plane),")
print("   which is what we expect from a matrix that projects onto the x-y plane.\n")
print("null space of N:\n", null_N)
print("-> the entire z axis: exactly the direction N throws away.")

### Which space is the null space perpendicular to?

It is tempting to draw the null vector sticking out of the column-space plane. **That is
wrong**, and the numbers below say so. The null space is defined by $Mu = 0$, and the
$i$-th entry of $Mu$ is (row $i$ of $M$) $\cdot\, u$. So $Mu = 0$ says precisely that $u$ is
orthogonal to **every row** of $M$:

$$\text{null space} \perp \text{row space}$$

There is no general relationship at all between the null space and the *column* space. For a
symmetric matrix the rows and columns are the same vectors and the two statements coincide —
which is probably why the confusion is so common — but $M$ here is not symmetric.

Two panels below: on the left the column space, where the outputs $Mv_1$ and $Mv_2$ live;
on the right the row space, to which the null vector is genuinely perpendicular. Both use
identical axis limits.

In [ ]:
# Row space basis: the column space of M transposed.
row_M = sla.orth(Mproj.T)

fig = plt.figure(figsize=(11, 5))
lim = (-2.2, 2.2)

ax = make_3d_axes(fig, 121, lim=lim, elev=20, azim=30)
draw_plane(ax, col_M, half=2.2)
plot_3d_vector(2*col_M[:, 0], "k", ax, label="column space basis")
plot_3d_vector(2*col_M[:, 1], "k", ax)
plot_3d_vector(u1, "tab:blue", ax, label="$Mv_1$")
plot_3d_vector(u2, "tab:red",  ax, label="$Mv_2$")
plot_3d_vector(2*null_M[:, 0], "tab:green", ax, label="null space")
ax.set_title("COLUMN space: outputs live here.\nThe null vector is NOT perpendicular to it.",
             fontsize=9)
ax.legend(loc="upper left", fontsize=7)

ax = make_3d_axes(fig, 122, lim=lim, elev=20, azim=30)
draw_plane(ax, row_M, half=2.2, color="tab:orange", alpha=0.16)
plot_3d_vector(2*row_M[:, 0], "tab:orange", ax, label="row space basis")
plot_3d_vector(2*row_M[:, 1], "tab:orange", ax)
plot_3d_vector(2*null_M[:, 0], "tab:green", ax, label="null space")
ax.set_title("ROW space: the null vector IS perpendicular to it.", fontsize=9)
ax.legend(loc="upper left", fontsize=7)
fig.tight_layout()

print("angle between the two column-space basis vectors: %.2f deg (orthonormal)"
      % np.degrees(np.arccos(np.clip(col_M[:, 0] @ col_M[:, 1], -1, 1))))
print()
print("null . COLUMN space basis: %+.4f, %+.4f   <- NOT zero"
      % (null_M[:, 0] @ col_M[:, 0], null_M[:, 0] @ col_M[:, 1]))
print("null . ROW    space basis: %+.2e, %+.2e   <- zero, as it must be"
      % (null_M[:, 0] @ row_M[:, 0], null_M[:, 0] @ row_M[:, 1]))
print()
for i in range(3):
    print("row %d of M . null vector = %+.2e" % (i, Mproj[i] @ null_M[:, 0]))

The dot products make it plain: the null vector is *not* orthogonal to the column-space basis
(dot products of about $+0.26$ and $-0.43$), but it *is* orthogonal to every row of $M$, to
machine precision.

So: we applied $M$ to two vectors and the results landed on a plane. That plane **is** the
column space of $M$, and `orth` gave us a convenient orthonormal coordinate system for it.
A separate plane — the row space — is what the null direction is perpendicular to. Keeping
these four spaces straight (column, row, null, left-null) is what Strang calls the
"fundamental theorem of linear algebra".

### The columns really do span the column space

A last sanity check, in the spirit of `ColumnSpaceDemo.m`: draw the columns of a matrix as
vectors. For a full-rank matrix they span the whole space; for a rank-deficient one they are
coplanar.

⚠️ A wrinkle worth knowing: the matrix in `ColumnSpaceDemo.m`,
$\left(\begin{smallmatrix}3&-2&1\\2&1&3\\-1&2&1\end{smallmatrix}\right)$, is itself
**singular** — its determinant is zero and its rank is 2, because its third column is the
sum of the first two. As a demonstration that three columns can fail to span a 3-D space it
is perfect; as a demonstration of the opposite it would not work. We use the invertible
$M$ from Part II for the full-rank panel.

In [ ]:
Mcsd = np.array([[ 3, -2, 1],
                 [ 2,  1, 3],
                 [-1,  2, 1]], dtype=float)   # ColumnSpaceDemo.m -- rank 2!

print("ColumnSpaceDemo.m matrix: det = %.4f, rank = %d"
      % (abs(np.linalg.det(Mcsd)), np.linalg.matrix_rank(Mcsd)))
print("   column 1 + column 2 =", Mcsd[:, 0] + Mcsd[:, 1],
      " = column 3 =", Mcsd[:, 2], "\n")

fig = plt.figure(figsize=(11, 5))
lim = (-3.4, 3.4)          # identical limits on both panels
for k, (A, name) in enumerate([(M, "M from Part II"),
                               (Mcsd, "the ColumnSpaceDemo.m matrix")], start=1):
    ax = make_3d_axes(fig, 120 + k, lim=lim, elev=20, azim=35)
    r = np.linalg.matrix_rank(A)
    if r == 2:
        draw_plane(ax, sla.orth(A), half=3.2)
    for j, col in enumerate(["tab:blue", "tab:green", "tab:red"]):
        plot_3d_vector(A[:, j], col, ax, label=f"column {j+1}")
    span = "columns span all of 3-D space" if r == 3 else "columns are coplanar"
    det = np.linalg.det(A)
    det = 0.0 if abs(det) < 1e-12 else det          # kill the ugly "-0.00"
    ax.set_title("%s\nrank %d, det = %.2f -- %s" % (name, r, det, span), fontsize=9)
    ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()

print("column 3 of Mproj as a combination of columns 1 and 2:")
wls, *_ = np.linalg.lstsq(Mproj[:, :2], Mproj[:, 2], rcond=None)
print("   c3 = %.4f*c1 + %.4f*c2 ; residual = %.2e"
      % (wls[0], wls[1], np.linalg.norm(Mproj[:, :2] @ wls - Mproj[:, 2])))

In the left panel the three columns point in genuinely independent directions and there is no
plane that contains all three. In the right panel all three lie in the shaded plane.

The same is true of `Mproj`: its third column is $1.0\,c_1 - 0.5\,c_2$, with a residual at
machine precision. That is the concrete meaning of "dependent columns", and it is why the
rank is 2 and the inverse does not exist.

> ### Homework question 3
> **(a)** Give an example of a matrix of rank 1. What are its column space and null space?
>
> **(b)** How many linearly independent vectors are represented in a matrix of rank 5?
>
> **(c)** Verify the rank–nullity theorem on a few matrices of your own, including a
> non-square one (say $4 \times 2$ and $2 \times 4$). What are the dimensions of the column
> and null spaces in each case?
>
> **(d)** Show from the definition $Mu = 0$ that the null space is orthogonal to the row
> space. Then construct a matrix for which the null space happens to *also* be orthogonal to
> the column space, and say what property of that matrix makes it work.
>
> **(e)** A matrix also has a **left null space**: the vectors $y$ with $y^{\mathsf{T}}M = 0$.
> Which of the other three spaces is it orthogonal to? Compute it for `Mproj` using
> `my_orth_null(Mproj.T)` and check your answer.

---
## Part V. The eigensystem and "natural" coordinates

Above we saw that the columns of a matrix define a space, and that matrices transform
vectors — rotating them, projecting them, scaling them. A key further idea is that matrices
can *represent coordinate systems*. The most useful example of this is a matrix's
**eigensystem**.

The eigenvalues $\lambda$ and eigenvectors $e$ of a matrix $M$ satisfy

$$M e = \lambda e$$

This matters because eigenvectors are **rescaled in length but not changed in direction** by
$M$. They therefore provide a natural coordinate system for $M$: one in which the operation
performed by $M$ is far simpler to describe than in any other choice of coordinates. Put
another way, $M$ does not *mix* the quantities described by different eigenvectors. An
$n \times n$ matrix has $n$ eigenvalues (counting multiplicity).

You can find eigenvalues analytically from the determinant of $M - \lambda I$. (It was
procedures like this, and inverting matrices by hand, that made me hate linear algebra the
first time I encountered it.) For more see Strang. In practice you find them numerically.

In [ ]:
Meig = np.array([[1., -1.,  0.],
                 [0.,  3., -1.],
                 [0.,  0.,  2.]])

# MATLAB: [EigVec, EigVal] = eig(M) -- EigVal is a DIAGONAL MATRIX.
# NumPy:  w, V = np.linalg.eig(M)   -- w is a 1-D ARRAY of eigenvalues,
#                                      V's COLUMNS are the eigenvectors.
w, V = np.linalg.eig(Meig)

print("eigenvalues :", w)
print("eigenvectors (columns):\n", V)
print("\nMATLAB's diagonal EigVal would be:\n", np.diag(w))

⚠️ **Three things to know about `np.linalg.eig`.**

1. **The order is not guaranteed.** LAPACK returns them in whatever order the algorithm
   produced. Never assume `w[0]` is the largest. If you need an order, sort explicitly:
   `idx = np.argsort(w)[::-1]; w, V = w[idx], V[:, idx]`.
2. **The values may be complex** even for a real matrix (rotations are the obvious case).
   `np.linalg.eig` returns a complex dtype whenever that can happen. For a **symmetric**
   matrix, use `np.linalg.eigh` instead: it exploits the symmetry, is faster and more
   accurate, guarantees real eigenvalues, guarantees orthogonal eigenvectors, and returns
   them in **ascending** order. Covariance matrices are symmetric, so `eigh` is the right
   call throughout the PCA tutorial.
3. **Sign and scale are arbitrary.** Both MATLAB and NumPy normalize to unit length, but the
   sign is whatever came out of LAPACK. A figure that flips when you rerun it on another
   machine is almost always this.

Let's check that the definition holds. Pull out one eigenvector and multiply by $M$.

In [ ]:
for k in range(3):
    e   = V[:, k]
    Me  = Meig @ e
    lam = w[k]
    print(f"--- eigenvector {k}  (MATLAB would call this EigVec(:,{k+1})) ---")
    print("   e      =", e)
    print("   M @ e  =", Me)
    print("   lam*e  =", lam * e, f"   (lambda = {lam:.4f})")
    print("   equal  :", np.allclose(Me, lam * e), "\n")

print("Note the eigenvalues 1, 3, 2 are exactly the DIAGONAL entries of M.")
print("That is a general fact for triangular matrices -- M here is upper triangular.")

### Eigenvectors as natural coordinates: a picture

Here is what "rescaled but not rotated" looks like. We apply a symmetric matrix

$$A = \begin{pmatrix}2 & 0.8\\ 0.8 & 1\end{pmatrix}$$

to a ring of unit vectors. Generic vectors get both stretched *and* turned; the two
eigenvectors get stretched only.

In [ ]:
A = np.array([[2.0, 0.8],
              [0.8, 1.0]])

# A is SYMMETRIC -> use eigh: real eigenvalues, orthonormal eigenvectors,
# ascending order (so we reverse to put the largest first).
lam, E = np.linalg.eigh(A)
order = np.argsort(lam)[::-1]
lam, E = lam[order], E[:, order]

# EIGENVECTOR SIGN IS ARBITRARY. LAPACK handed back e1 pointing down-left here;
# flipping it to point up-right changes nothing mathematically and makes the
# figure readable. Doing this deliberately is fine; depending on the sign you
# happen to get is not.
E = E * np.sign(E[np.argmax(np.abs(E), axis=0), np.arange(2)])

th = np.linspace(0, 2*np.pi, 24, endpoint=False)
circle = np.vstack([np.cos(th), np.sin(th)])       # 2 x 24, unit vectors
mapped = A @ circle

fig, ax = plt.subplots(figsize=(6.5, 6.5))
for j in range(circle.shape[1]):
    ax.plot([circle[0, j], mapped[0, j]], [circle[1, j], mapped[1, j]],
            "-", color="0.78", lw=0.8, zorder=1)
ax.plot(*circle, "o", ms=3.5, color="tab:blue", label="unit vectors $v$", zorder=3)
ax.plot(*mapped, "o", ms=3.5, color="tab:red",  label="$A\\,v$", zorder=3)
for k, c in enumerate(["k", "tab:green"]):
    e = E[:, k]
    ax.plot([-3*e[0], 3*e[0]], [-3*e[1], 3*e[1]], ":", color=c, lw=1, zorder=2)
    plot_vector(e,          c, ax, lw=3,
                label=f"eigenvector {k+1} ($\\lambda$={lam[k]:.2f})")
    plot_vector(lam[k] * e, c, ax, lw=1.4, ls="--")
ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal",
       xlabel="x", ylabel="y",
       title="$A$ stretches the eigen-directions and rotates everything else\n"
             "(dotted = invariant axes; solid = $e$, dashed = $\\lambda e$)")
ax.legend(loc="lower left", fontsize=8)
fig.tight_layout()

print("eigenvalues:", lam.round(4))
for k in range(2):
    e = E[:, k]
    ang = np.degrees(np.arccos(np.clip(e @ (A @ e) / np.linalg.norm(A @ e), -1, 1)))
    print(f"eigenvector {k+1}: angle between v and Av = {ang:.2e} deg (unchanged direction)")
gen = np.array([1.0, 0.0])
ang = np.degrees(np.arccos(gen @ (A @ gen) / np.linalg.norm(A @ gen)))
print(f"generic vector (1,0): angle between v and Av = {ang:.2f} deg (rotated)")

The grey lines show where each unit vector moves. Only along the two eigen-directions does a
vector move straight out along its own line — everywhere else it swings around. The dashed
segments show $\lambda e$, the image of each eigenvector.

Because $A$ is symmetric, its eigenvectors came out orthogonal; that is guaranteed by the
spectral theorem and is why `eigh` exists. For a non-symmetric matrix (such as the transition
matrix in Part VI) the eigenvectors are generally **not** orthogonal, and this picture would
look skewed.

### Reconstructing the matrix from its eigensystem

If the eigenvectors are independent, we can collect them as columns of $E$ and write

$$M = E \Lambda E^{-1}$$

with $\Lambda = \mathrm{diag}(\lambda_1,\dots,\lambda_n)$. This is the **eigendecomposition**,
and it is what makes Part VI work: powers and exponentials of $M$ become powers and
exponentials of the *scalars* on the diagonal.

In [ ]:
w, V = np.linalg.eig(Meig)
recon = V @ np.diag(w) @ np.linalg.inv(V)
print("E Lambda E^-1 =\n", recon.real)
print("original M    =\n", Meig)
print("agree:", np.allclose(recon.real, Meig), "\n")

# and the payoff: M^8 the hard way vs via the eigensystem
Mpow_direct = np.linalg.matrix_power(Meig, 8)     # MATLAB: M^8
Mpow_eigen  = (V @ np.diag(w**8) @ np.linalg.inv(V)).real
print("M^8 by repeated multiplication:\n", Mpow_direct)
print("M^8 via E diag(lam^8) E^-1   :\n", Mpow_eigen)
print("agree:", np.allclose(Mpow_direct, Mpow_eigen))
print("\n(NOTE: in NumPy `Meig ** 8` is ELEMENTWISE, not the matrix power.)")
print("Meig ** 8 (wrong!) =\n", Meig ** 8)

> ### Homework question 4
> **(a)** Verify $Me = \lambda e$ for the remaining eigenvectors of `Meig` by hand for at
> least one case, to convince yourself NumPy is not making it up.
>
> **(b)** `Meig` is upper triangular and its eigenvalues turned out to be its diagonal
> entries. Prove this is true for any triangular matrix. *Hint: what does
> $\det(M - \lambda I)$ look like for a triangular matrix?*
>
> **(c)** Compute the eigenvalues of the 2-D rotation matrix `R` from Part III. Why are they
> complex, and what do their modulus and argument correspond to geometrically?
>
> **(d)** What are the eigenvalues and eigenvectors of a projection matrix such as $N$? What
> does an eigenvalue of 0 mean, and how does it relate to the null space?

---
## Part VI *(optional)*. Markov chains and state-transition matrices

> **This section is optional.** Skip it unless you (a) are looking for a way to burn some
> time, (b) already know something about Markov processes, and (c) are comfortable with the
> eigensystem material above. Otherwise wait — we come back to these ideas at the end of the
> course. See the Markov model notes on the course website.

Markov transition matrices are a nice example of the utility of eigenvectors and
eigenvalues. Here the eigensystem identifies **modes** of the system, each with its own
characteristic kinetics, which lets us solve the dynamics of a set of coupled states quite
simply.

We work through an ion channel with three states: closed (C), open (O) and inactivated (I),
linked by rate constants. The only permitted transitions are

$$\text{closed} \longleftrightarrow \text{open} \longleftrightarrow \text{inactivated}$$

We write the probability of being in each state at time $t + \Delta t$ in terms of the
probabilities at time $t$ and a transition matrix holding the rate constants. We need an
initial state, which we take to be "all channels closed".

In [ ]:
InitialS = np.array([1.0, 0.0, 0.0])      # closed, open, inactivated

# Markov transition matrix. Column j says where probability in state j goes.
# Diagonal entries = probability of STAYING; off-diagonals = moving.
Mk = np.array([[0.95, 0.10, 0.00],
               [0.05, 0.75, 0.01],
               [0.00, 0.15, 0.99]])

# This assumes  dt = 1 msec  and the rate constants
#   closed      -> open           50 /sec
#   open        -> closed        100 /sec
#   open        -> inactivated   150 /sec
#   inactivated -> open           10 /sec
#   closed     <-> inactivated   disallowed

print("Mk =\n", Mk)
print("\ncolumn sums (must each be 1, so probability is conserved):", Mk.sum(axis=0))
print("\nS after one 1 ms step, Mk @ InitialS =", Mk @ InitialS)

### Approach 1: brute force

We can iterate this recursively to get the state after $N$ steps, i.e. at time
$N \Delta t$.

⚠️ In MATLAB `M^N` is the matrix power. In NumPy `Mk ** N` raises each *element* to the
power $N$ — a silent, badly wrong answer. Use `np.linalg.matrix_power`.

In [ ]:
N = 10                                        # 10 steps = 10 msec
S10 = np.linalg.matrix_power(Mk, N) @ InitialS

fig = plt.figure(figsize=(6, 5.5))
ax = make_3d_axes(fig, lim=(0, 1), elev=18, azim=40,
                  labels=["closed", "open", "inactivated"])
plot_3d_vector(InitialS, "tab:blue", ax, label="$S(0)$")
plot_3d_vector(S10,      "tab:red",  ax, label="$S(10\\,\\mathrm{ms}) = M^{10}S(0)$")
ax.set_title("state probability vector, brute force")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()

print("S(10 ms) =", S10, "  (sums to %.6f)" % S10.sum())

This lets us find the state after any number of steps, but it is brute force: it gives
little intuition about *how* the system works.

### Approach 2: the eigensystem

First rewrite the transition matrix to separate out the probability of *not* changing state
and to pull out the time step. Define a rate matrix $T$ such that

$$M = I + \Delta t\, T$$

In [ ]:
T = np.array([[ -50., 100.,   0.],
              [  50., -250., 10.],
              [   0.,  150., -10.]])

DeltaT = 0.001
Identity = np.eye(3)

print("I + dt*T =\n", Identity + DeltaT * T)
print("\nMk      =\n", Mk)
print("\nagree:", np.allclose(Identity + DeltaT * T, Mk))
print("\ncolumn sums of T (must be 0 -- rate in equals rate out):", T.sum(axis=0))

Now the update rule takes a form we can interpret with what we know about differential and
difference equations. The new state at $t+\Delta t$ in terms of the state at $t$ is

$$S(t + \Delta t) = S(t) + \Delta t\, T\, S(t)$$

Rearranging,

$$\frac{S(t+\Delta t) - S(t)}{\Delta t} = T\,S(t)$$

The left side is the difference-equation version of the derivative, so this is

$$\frac{dS}{dt} = T\,S(t)$$

and the only complication is that $T$ is a matrix and $S$ a vector. If $S$ were a single
variable and $T$ a scalar, we would guess $S(t) = A e^{Tt}$.

**What if $S$ were an eigenvector of $T$?** Then $TS = \lambda S$ and

$$\frac{dS}{dt} = \lambda S(t)$$

which is just three copies of the scalar equation $dx/dt = ax$, whose solution is
$S(t) = S(0)e^{\lambda t}$.

How does that help in general? Expand $S(t)$ in the eigenvectors of $T$:

$$S(t) = \sum_n a_n(t)\, e_n$$

This is a series expansion, like the Fourier expansion in sines and cosines, but with our
eigenvectors as the basis. Then

$$\frac{dS}{dt} = \sum_n e_n \frac{da_n}{dt}, \qquad
  T S(t) = \sum_n a_n(t)\, T e_n = \sum_n a_n(t)\, \lambda_n e_n$$

Equating the two and matching coefficients of each $e_n$ gives

$$\frac{da_n}{dt} = \lambda_n a_n(t) \quad\Longrightarrow\quad
  a_n(t) = a_n(0)\, e^{\lambda_n t}$$

⚠️ **A correction to the MATLAB original.** The `.m` file says we "use the orthogonality of
the $e_n$" to pull out each $a_n$. That is not right here: $T$ is **not symmetric**, so its
eigenvectors are **not** orthogonal — we check this explicitly below. What actually lets us
separate the coefficients is that the $e_n$ are *linearly independent*, so they form a
basis, and the expansion coefficients are unique. Concretely you get them by inverting the
eigenvector matrix, which is precisely what the code does (`a = inv(EigVec) * InitialS`).
Equivalently, the rows of $E^{-1}$ are the **left** eigenvectors of $T$, and left and right
eigenvectors *are* mutually orthogonal across different eigenvalues (bi-orthogonality). The
orthogonality argument works verbatim only for symmetric matrices — which is why it is
legitimate in the PCA tutorial, where the matrix is a covariance matrix.

In [ ]:
lamT, E = np.linalg.eig(T)

# T is real but not symmetric, so eig() may return a complex dtype. Here the
# eigenvalues happen to be real; drop the zero imaginary part explicitly rather
# than letting it propagate silently through the plotting code.
assert np.allclose(lamT.imag, 0) and np.allclose(E.imag, 0)
lamT, E = lamT.real, E.real

order = np.argsort(lamT)                  # eig() does NOT sort; do it ourselves
lamT, E = lamT[order], E[:, order]

print("eigenvalues of T :", lamT)
print("time constants -1/lambda (sec):",
      np.array([np.inf if abs(l) < 1e-9 else -1/l for l in lamT]).round(5))
print("\neigenvectors (columns):\n", E)

print("\nAre the eigenvectors orthogonal?  E^T E should be the identity if so:")
print(np.round(E.T @ E, 4))
print("-> clearly NOT the identity. T is not symmetric, so they are not orthogonal.")

print("\nBi-orthogonality instead: inv(E) @ E =")
print(np.round(np.linalg.inv(E) @ E, 6))

So the eigenvalues are about $-278$, $-32$ and $0$ per second — time constants of roughly
**3.6 ms** and **31 ms**, plus one mode that does not decay at all. A few observations:

- With only two nonzero eigenvalues, $T$ has **rank 2**.
- The nonzero eigenvalues are **negative**, so the associated coefficients $a_n(t)$ *decay*
  exponentially to zero rather than blowing up. That is a good thing: otherwise we could not
  maintain $\sum_i S_i = 1$.
- $T$ **has** to be rank 2 (i.e. to have one zero eigenvalue). If all three eigenvalues were
  negative, every $a_n(t)$ would decay and we would be left with nothing — again breaking
  the requirement that the state probabilities sum to 1. The zero-eigenvalue mode is the
  **steady state**.

Our initial $S$ was $(1, 0, 0)$. To write it as a weighted sum of eigenvectors we want
$S(0) = E\,a$, so $a = E^{-1}S(0)$. This is the inverse used exactly as in the
three-equations-three-unknowns problem in Part II (and exactly as when generating
cone-isolating stimuli in class).

In [ ]:
# a = inv(EigVec) * InitialS, but computed with solve -- see the Part II note.
a = np.linalg.solve(E, InitialS)
print("expansion coefficients a =", a)
print("check, E @ a =", E @ a, " should be InitialS =", InitialS)
print("agree:", np.allclose(E @ a, InitialS), "\n")

# S(t) = sum_n a_n exp(lambda_n t) e_n  -- vectorized over the 3 modes
tme = 0.01                                       # 10 msec
S_eig = (E * (a * np.exp(lamT * tme))).sum(axis=1)

print("S(10 ms) from the eigensystem :", S_eig)
print("S(10 ms) from M^10 (approach 1):", S10)
print("S(10 ms) exact, expm(T*t)     :", sla.expm(T * tme) @ InitialS)
print("\neigensystem vs expm agree:",
      np.allclose(S_eig, sla.expm(T * tme) @ InitialS))
print("max |eigensystem - M^10| = %.4f" % np.max(np.abs(S_eig - S10)))

The eigensystem result and `scipy.linalg.expm` agree to machine precision — as they must,
since $e^{Tt} = E\,e^{\Lambda t}E^{-1}$ is exactly what we computed by hand.

The brute-force $M^{10}$ answer differs by about 0.005 in the largest component. That is not
an error in either calculation: they are answers to slightly different questions. $M^N$
solves the *discrete-time* system with $\Delta t = 1$ ms, whereas $e^{Tt}$ solves the
*continuous-time* differential equation. They converge as the step shrinks, because

$$\left(I + \frac{Tt}{m}\right)^{m} \longrightarrow e^{Tt} \quad\text{as } m \to \infty$$

which is the matrix version of the familiar scalar limit $(1 + x/m)^m \to e^x$. That limit
is the point of the course's `expAsLimit.m`; let's watch it converge for both.

In [ ]:
# --- scalar version (expAsLimit.m) and the matrix version, side by side ---
k = 0.1
m = np.arange(1, 101)
y = (1 - k/m)**m                                   # -> exp(-k)

t_total = 0.01                                     # 10 msec
steps = np.unique(np.round(np.logspace(0, 3.5, 40)).astype(int))
exact = sla.expm(T * t_total) @ InitialS
err = np.array([
    np.max(np.abs(np.linalg.matrix_power(np.eye(3) + T*(t_total/mm), mm) @ InitialS
                  - exact))
    for mm in steps])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
axes[0].plot(m, y, "k-", lw=2, label=r"$(1 - k/m)^m$")
axes[0].axhline(np.exp(-k), color="tab:red", lw=1.5, ls="--",
                label=r"$e^{-k}$ = %.5f" % np.exp(-k))
axes[0].set(xlabel="m", ylabel="y", title="scalar: $(1-k/m)^m \\to e^{-k}$,  k = 0.1")
axes[0].legend(fontsize=9)

axes[1].loglog(steps, err, "o-", color="tab:blue", ms=4)
axes[1].axvline(10, color="0.5", ls=":", lw=1)
axes[1].text(10.5, err.max()*0.4, "m = 10\n(the 1 ms step\nused above)", fontsize=8)
axes[1].set(xlabel="number of steps m over 10 ms",
            ylabel=r"max $|(I+Tt/m)^m S_0 - e^{Tt}S_0|$",
            title="matrix: the same limit, first order in $1/m$")
axes[1].grid(True, which="both", alpha=0.3)
fig.tight_layout()

print("scalar:  (1-k/m)^m at m=100 = %.6f ;  exp(-k) = %.6f" % (y[-1], np.exp(-k)))
print("matrix:  error at m=10   = %.5f" % err[np.argmin(np.abs(steps-10))])
print("         error at m=%d = %.2e" % (steps[-1], err[-1]))

Both panels show the same thing: the discrete update is a first-order approximation to the
continuous one, and the error falls roughly as $1/m$. With $m = 10$ steps over 10 ms the
error is a few parts in a thousand, which is exactly the discrepancy we saw above.

### The full time course

We can now solve for $S(t)$ at *any* time in one shot, with no iteration.

In [ ]:
tme = np.arange(1, 1001) * 0.001                   # 1 ms .. 1 sec

# S(t) = sum_n a_n exp(lam_n t) e_n, all times and modes at once.
#   np.exp(np.outer(tme, lamT)) is (1000, 3);  E is (3, 3)
# MATLAB does this with a nested double loop over n and m -- same arithmetic.
S = (a * np.exp(np.outer(tme, lamT))) @ E.T        # (1000, 3)

names = ["closed", "open", "inactivated"]
colors = ["tab:blue", "tab:green", "tab:red"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for j in range(3):
    axes[0].plot(tme, S[:, j], color=colors[j], lw=2, label=names[j])
    axes[0].axhline(S[-1, j], color=colors[j], lw=0.8, ls=":")
axes[0].set(xlabel="time (sec)", ylabel="probability", ylim=(0, 1),
            title="state probabilities (dotted = steady state)")
axes[0].legend(fontsize=9)

for j in range(3):
    axes[1].semilogy(tme*1000, np.abs(S[:, j] - S[-1, j]) + 1e-18,
                     color=colors[j], lw=2, label=names[j])
axes[1].set(xlabel="time (msec)", ylabel="|S(t) - S(inf)|", xlim=(0, 200),
            ylim=(1e-6, 1),
            title="approach to steady state: two exponentials")
axes[1].legend(fontsize=9)
fig.tight_layout()

print("probability sums to 1 at every time:",
      np.allclose(S.sum(axis=1), 1.0))
print("S at t = 1 sec:", S[-1].round(4))

On a log scale (right panel) the approach to steady state is close to a straight line — the
signature of exponential decay. Two exponentials are present, with the fast one
($\tau \approx 3.6$ ms) visible only in the first few milliseconds and the slow one
($\tau \approx 31$ ms) dominating thereafter.

So what does this tell us? We have identified characteristic **modes** of the system —
particular combinations of the three states, each of which decays with its own exponential
rate constant. We could trace those rate constants back to the original transition rates and
so get a feel for what governs the behavior. That is exactly the intuition brute-force
iteration does not give you.

### The steady state, three ways

The steady state is the eigenvector with eigenvalue 0, normalized so the probabilities sum
to 1. We can get the same numbers from detailed balance and from the discrete chain, which
is a good cross-check.

In [ ]:
# (1) from the zero-eigenvalue eigenvector of T
i0 = int(np.argmin(np.abs(lamT)))
ss_eig = E[:, i0] / E[:, i0].sum()          # sign/scale are arbitrary -> normalize

# (2) from detailed balance, by hand:
#     pC * 50 = pO * 100   ->  pC = 2 pO
#     pO * 150 = pI * 10   ->  pI = 15 pO
#     pC + pO + pI = 1     ->  18 pO = 1
pO = 1/18
ss_hand = np.array([2*pO, pO, 15*pO])

# (3) from the discrete chain, iterated a long way
ss_disc = np.linalg.matrix_power(Mk, 20000) @ InitialS

print("               closed     open   inactivated")
print("eigenvector : ", ss_eig.round(6))
print("by hand     : ", ss_hand.round(6))
print("discrete M^N: ", ss_disc.round(6))
print("\neigenvector vs hand agree:", np.allclose(ss_eig, ss_hand))
print("discrete vs hand agree   :", np.allclose(ss_disc, ss_hand, atol=1e-6))
print("\nexact fractions: 2/18, 1/18, 15/18 = %.4f, %.4f, %.4f"
      % (2/18, 1/18, 15/18))

# how long to get within 1% of steady state?
dev = np.max(np.abs(S - ss_hand), axis=1)
i99 = int(np.argmax(dev < 0.01 * ss_hand.max()))
print("\ntime for every state to come within 1%% of the largest steady-state "
      "value: %.0f msec" % (tme[i99]*1000))
print("compare the slow time constant, -1/lambda = %.1f msec" % (-1000/lamT[1]))
print("-> about %.1f slow time constants, as you would expect for a factor "
      "of ~%.0f decay." % (tme[i99] / (-1/lamT[1]), 1/0.01))

All three routes give the same answer: closed $= 1/9 \approx 0.111$, open
$= 1/18 \approx 0.056$, inactivated $= 5/6 \approx 0.833$. The channel spends most of its
time inactivated, which is what you would guess from the rate constants — the
open$\to$inactivated rate (150/s) is fifteen times the inactivated$\to$open rate (10/s).

Note that reaching the steady state takes a few times the *slow* time constant, not the
fast one. Whenever a system has widely separated time constants, the slow mode sets how long
you have to wait.

> ### Homework question 5 *(for your own edification — not to be turned in)*
> **(a)** Why does the Markov transition matrix have the form it does? In particular, why
> must every column sum to 1, and why must every column of $T$ sum to 0?
>
> **(b)** What are the steady-state probabilities of being closed, open and inactivated?
> Derive them by hand from the rate constants and check against the code.
>
> **(c)** How long does this channel model take to get within 1% of the steady-state values?
> Can you explain where that time scale comes from?
>
> **(d)** The MATLAB original invokes orthogonality of the $e_n$ to extract the $a_n(t)$
> coefficients. Show why that argument fails for a non-symmetric $T$, and give the correct
> derivation using $E^{-1}$ (equivalently, the left eigenvectors).
>
> **(e)** Confirm that both approaches above yield the same steady-state values for each
> channel state, and explain why the discrete chain converges to the same place as the
> continuous one even though $M^{10}$ and $e^{0.01T}$ differ.
>
> **(f)** Change the open$\to$inactivated rate from 150/s to 15/s. Predict, before running
> anything, how the steady state and the two time constants will change. Then check.

---
## Summary

1. **Vectors are ordered collections of numbers**, and the axes need not be spatial or even
   share units. Representing data as vectors is what makes it visualizable and what lets
   matrices act on it.

2. **A matrix is an operator.** It scales, rotates, reflects or projects the vectors it acts
   on. Reading a matrix as "what does this do to a vector?" is far more useful than reading
   it as a table of numbers.

3. **The inverse undoes a transformation**, and it converts "$n$ equations in $n$ unknowns"
   into a single matrix operation, $u = M^{-1}v$. In practice never form the inverse: use
   `np.linalg.solve` (MATLAB's `\`), which runs the Gaussian elimination we wrote out by
   hand and is both faster and more accurate.

4. **Not every matrix has an inverse.** A matrix with dependent columns projects many
   vectors onto one, and the information destroyed cannot be recovered. The **rank** counts
   the independent directions that survive; the **column space** is where outputs live; the
   **null space** is what gets annihilated; and $\text{rank} + \text{nullity} = $ the number
   of columns.

5. **Eigenvectors are the natural coordinates of a matrix.** $Me = \lambda e$ means $M$
   rescales those directions without mixing them, which turns a coupled problem into a set
   of independent scalar ones. Remember that eigenvalue *order* is arbitrary, eigenvalues
   may be complex, eigenvector *sign and scale* are arbitrary, and that orthogonality of
   eigenvectors is guaranteed only for symmetric matrices (use `eigh` there).

6. **In the Markov example the eigensystem pays off concretely**: the eigenvalues of the rate
   matrix are the decay rates of the system's modes, the zero eigenvalue is the steady state,
   and the whole time course comes out in closed form instead of by iteration.

These ideas recur throughout the course. Differential equations use the eigensystem to
decouple modes exactly as we did in Part VI. PCA is the eigendecomposition of a covariance
matrix — a symmetric one, so `eigh` applies and the eigenvectors are genuinely orthogonal.
Regression and the SVD tutorial pick up the story of what to do when the matrix is not
invertible or not even square.

### Further reading

- Strang, G. (2016). *Introduction to Linear Algebra*, 5th ed. — the standard reference;
  chapters 1–6 cover everything here.
- Simoncelli, E. *A Geometric View of Linear Algebra* (the Linear Algebra Primer on the
  course website) — short, and matched to the way this tutorial thinks.
- Strang's MIT 18.06 lectures, freely available on MIT OpenCourseWare.
- Trefethen & Bau (1997). *Numerical Linear Algebra* — why `solve` beats `inv`, and what
  conditioning really means.
- In this course, see also the color space tutorial, the SVD tutorial, the regression
  tutorial, `PCATutorial.m` and `PCANeuroPopTutorial.m`.